# AB Testing Assignment: Optimizing Engagement in a Mental Wellness App

Your Name

**Estimated Time:** 60 minutes | **Difficulty:** Intermediate

---

## Getting Started

- Replace **Your Name** above with your full name
- Automatic 0 if you include your student ID or any other identifying number
- Rename the file to `Your_Name.ipynb` before submitting
- When finished, share your Colab link with **Anyone with the link can view** privileges
- Paste the shared link into the Canvas submission

---

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Every code section in this notebook gives you a choice. Before each code cell you will find a navigation landmark with two options:

- **Skip this code cell** — Jumps past the code directly to the output explanation. Use this to focus on understanding results without reading raw code.
- **Go back and read the code cell** — Returns you to the top of the code section if you want to review it.

Each code cell is preceded by a plain-English description of what the code does, and followed by a description of the expected output.

**Tips for screen reader users:**
- Press **H** to jump between major section headings
- Press **K** or **Tab** to move between links, including skip and return navigation links
- Press **D** to jump between landmark regions

---

## Learning Objectives

By the end of this assignment you will be able to:

1. Design a complete AB test from hypothesis formulation through business recommendation
2. Formulate null and alternative hypotheses for a real-world scenario
3. Calculate and interpret test statistics, p-values, and critical values
4. Construct and interpret confidence intervals for the difference between two proportions
5. Distinguish between statistical significance and practical significance
6. Calculate statistical power and understand its role in sample size decisions
7. Communicate statistical results clearly to a non-technical audience

---

## Deliverables

| Part | Title | Time |
| :--- | :--- | :--- |
| **Part 1** | Setup and Data Generation | 10 min |
| **Part 2** | Exploratory Data Analysis | 10 min |
| **Part 3** | Hypothesis Testing | 20 min |
| **Part 4** | Confidence Intervals | 10 min |
| **Part 5** | Business Interpretation and Practical Significance | 10 min |
| **Part 6** | Power Analysis and Sample Size (Bonus) | — |
| **Part 7** | Final Report and Recommendations | — |

---

## Business Scenario

You are a data scientist at **MindPath**, a fast-growing mental wellness app that helps users build daily emotional check-in habits. Users open the app each day, answer a brief mood questionnaire, and receive personalized wellness insights based on their responses.

The product team has developed a new AI-personalized push notification system. Instead of sending every user the same generic reminder ('It's time for your daily check-in!'), the new system analyzes each user's past behavior — the times they typically engage, their mood patterns, and their stated wellness goals — and sends a message tailored specifically to them.

### The Question
**Does the AI-personalized notification increase the daily check-in completion rate compared to the generic notification?**

### Current Performance
- The generic notification has a historical check-in completion rate of approximately **28%**
- The product team wants at least a **3 percentage point improvement** (to 31%) to justify the engineering investment
- They want to be 95% confident in the results (α = 0.05)

### The Test Setup
- **Control Group (A):** Users receive the existing generic reminder notification
- **Treatment Group (B):** Users receive the new AI-personalized notification
- Users are randomly assigned to one group for a 4-week period
- The outcome is binary: did the user complete their daily check-in? (1 = yes, 0 = no)

---

# Part 1: Setup and Data Generation
**Estimated time: 10 minutes**

In this part you will set up the testing environment, formulate your hypotheses, and generate the simulated AB test data. Complete the written hypothesis task before running the data generation cell.

## Task 1.1: Import Libraries and Set Seed

<a name="read-code-1"></a>

**Cell 1 — Import Libraries and Generate Session Seed**

This cell imports all libraries needed for the assignment: `numpy` for numerical operations, `pandas` for data manipulation, `matplotlib` and `seaborn` for visualization, `scipy.stats` for statistical tests, and `datetime` for timestamps. It then generates a unique time-based seed and prints it. **Save your seed number** — you will need it to reproduce your exact results.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm, binom
import time
from datetime import datetime

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

seed = int(time.time() * 1000) % 100000
np.random.seed(seed)

print('='*70)
print('AB TEST SIMULATION — MINDPATH WELLNESS APP')
print('='*70)
print(f'Random Seed: {seed}')
print(f'Date/Time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'\n⚠️  IMPORTANT: Save this seed number to reproduce your exact results!')
print('='*70)

<a name="skip-code-1"></a>

**Expected output:** A header block showing your unique seed and the current timestamp. Copy the seed number into a comment or note — without it you cannot reproduce your exact dataset.

**Why a time-based seed?** It ensures every student's dataset is slightly different, which means your specific p-value and confidence interval will be unique to your session.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

## Task 1.2: Formulate Your Hypotheses

**Before collecting data, you must clearly state what you are testing.** This is one of the most important disciplines in statistical practice — changing your hypothesis after seeing the data (called HARKing: Hypothesizing After Results are Known) invalidates the entire test.

Let:
- p_A = daily check-in completion rate for the Control group (generic notification)
- p_B = daily check-in completion rate for the Treatment group (AI-personalized notification)

**✏️ YOUR TASK:** Complete the hypotheses below by editing this Markdown cell.

**Null Hypothesis (H₀):**
*[Write what you are testing against — hint: is there a difference between the two groups?]*

**Alternative Hypothesis (Hₐ):**
*[Write what the product team hopes to find — hint: does the personalized notification perform better?]*

**Significance Level (α):**
*[What is the threshold for rejecting H₀?]*

**Type of Test:**
*[One-tailed or two-tailed? Justify your choice.]*

---
**YOUR ANSWER HERE** — replace the italicized lines above.

## Task 1.3: Generate Simulated AB Test Data

<a name="read-code-2"></a>

**Cell 2 — Generate the AB Test Dataset**

This cell defines the true underlying check-in completion rates (which in a real test we would not know — we are simulating them). Control users have a 28% completion rate; treatment users have a 31.5% rate (a 3.5 percentage point lift). It draws 1,500 Bernoulli samples for each group using `np.random.binomial`, assembles them into a single DataFrame, and shuffles the rows to simulate random arrival order.

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# True underlying completion rates (unknown to analysts in a real test)
true_control_rate  = 0.28   # 28% baseline — generic notification
true_treatment_rate = 0.315  # 31.5% with AI-personalized notification

# Sample sizes
n_control   = 1500
n_treatment = 1500

# Generate completion data (1 = completed check-in, 0 = did not complete)
control_completions   = np.random.binomial(1, true_control_rate,   n_control)
treatment_completions = np.random.binomial(1, true_treatment_rate, n_treatment)

# Build DataFrames
df_control = pd.DataFrame({
    'user_id': range(1, n_control + 1),
    'group': 'Control',
    'completed': control_completions
})
df_treatment = pd.DataFrame({
    'user_id': range(n_control + 1, n_control + n_treatment + 1),
    'group': 'Treatment',
    'completed': treatment_completions
})

# Combine and shuffle
df = pd.concat([df_control, df_treatment], ignore_index=True)
df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

print('📊 Data Generated Successfully!')
print(f'Total users:     {len(df):,}')
print(f'Control group:   {len(df[df["group"]=="Control"]):,}')
print(f'Treatment group: {len(df[df["group"]=="Treatment"]):,}')
print('\nFirst 10 rows:')
print(df.head(10))

<a name="skip-code-2"></a>

**Expected output:** A summary showing 3,000 total users split evenly, followed by a 10-row preview with `user_id`, `group`, and `completed` (0 or 1) columns.

**Note:** Because we use a random seed, your completion totals will differ slightly each session. That is expected and reflects the natural sampling variation you will quantify in Parts 3 and 4.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 2: Exploratory Data Analysis
**Estimated time: 10 minutes**

Before running any statistical tests, always explore and visualize your data. EDA helps you catch data quality issues, understand the magnitude of any observed difference, and form initial intuitions that your formal test will later confirm or refute.

## Task 2.1: Calculate Sample Statistics

<a name="read-code-3"></a>

**Cell 3 — Calculate Group-Level Statistics**

This cell computes sample size, number of completions, and completion rate for each group, then calculates the observed difference (Treatment minus Control) in both absolute and relative terms. Two lines contain blanks you must fill in: the completion rate for each group is completions divided by sample size.

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Control group
n_A          = len(df[df['group'] == 'Control'])
completions_A = df[df['group'] == 'Control']['completed'].sum()
p_A          = ___  # YOUR CODE: completion rate = completions / sample size

# Treatment group
n_B          = len(df[df['group'] == 'Treatment'])
completions_B = df[df['group'] == 'Treatment']['completed'].sum()
p_B          = ___  # YOUR CODE: completion rate

# Observed difference
observed_diff = p_B - p_A

print('='*70)
print('SAMPLE STATISTICS')
print('='*70)
print(f'\nControl Group (A — Generic Notification):')
print(f'  Sample size:       {n_A:,}')
print(f'  Completions:       {completions_A:,}')
print(f'  Completion rate:   {p_A:.4f} ({p_A*100:.2f}%)')
print(f'\nTreatment Group (B — AI-Personalized Notification):')
print(f'  Sample size:       {n_B:,}')
print(f'  Completions:       {completions_B:,}')
print(f'  Completion rate:   {p_B:.4f} ({p_B*100:.2f}%)')
print(f'\nObserved Difference (B − A):')
print(f'  Absolute: {observed_diff:.4f} ({observed_diff*100:.2f} percentage points)')
print(f'  Relative: {(observed_diff/p_A)*100:.2f}% lift')
print('='*70)

<a name="skip-code-3"></a>

**Expected output:** Summary statistics for both groups. The Control completion rate should be close to 28% and Treatment close to 31.5%, with variation due to your random seed.

**If you see a NameError:** The `___` blanks were not filled in. Replace each `___` with `completions_A / n_A` and `completions_B / n_B` respectively.

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

## Task 2.2: Visualize the Results

<a name="read-code-4"></a>

**Cell 4 — Create Group Comparison Visualizations**

This cell produces a two-panel figure. The left panel shows a bar chart of completion rates per group with the 28% historical baseline marked. The right panel shows a stacked bar chart breaking down completions versus non-completions by group. Both panels use high-contrast colors and labeled axes.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: completion rates
groups = ['Control\n(Generic)', 'Treatment\n(AI-Personalized)']
rates  = [p_A, p_B]
colors = ['#3498db', '#8e44ad']

bars = ax1.bar(groups, rates, color=colors, alpha=0.75, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Check-In Completion Rate', fontsize=12)
ax1.set_title('Completion Rates by Group', fontsize=14, fontweight='bold')
ax1.set_ylim(0, max(rates) * 1.35)
for bar, rate in zip(bars, rates):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{rate*100:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax1.axhline(y=0.28, color='red', linestyle='--', linewidth=2, alpha=0.75, label='Historical Baseline (28%)')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Right panel: stacked breakdown
conv_data = pd.DataFrame({
    'Group': ['Control']*2 + ['Treatment']*2,
    'Status': ['Completed', 'Not Completed']*2,
    'Count': [completions_A, n_A - completions_A, completions_B, n_B - completions_B]
})
pivot_data = conv_data.pivot(index='Group', columns='Status', values='Count')
pivot_data.plot(kind='bar', stacked=True, ax=ax2, color=['#8e44ad', '#e74c3c'], alpha=0.75, edgecolor='black')
ax2.set_ylabel('Number of Users', fontsize=12)
ax2.set_title('Check-In Breakdown by Group', fontsize=14, fontweight='bold')
ax2.set_xlabel('')
ax2.set_xticklabels(['Control', 'Treatment'], rotation=0)
ax2.legend(title='Status')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_group_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-4"></a>

**Expected output:** A two-panel figure.

**Accessibility note:** Add a Markdown cell below this one describing both charts in plain language. Example: 'The left bar chart shows the Treatment group completion rate (approximately 31%) is higher than the Control group rate (approximately 28%), both above the 28% historical baseline. The right stacked bar chart shows similar total heights for both groups, with the Treatment bar having a slightly larger purple (completed) section.'

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

**🤔 Reflection Question:** Based on the visualizations, does it appear that the AI-personalized notification performs better? Write your initial thoughts below **before** conducting the formal statistical test.

**YOUR ANSWER HERE:**

*[Your initial observations about the charts — does the difference look meaningful? Could it be due to random chance?]*

---
# Part 3: Hypothesis Testing
**Estimated time: 20 minutes**

Now you will conduct a formal **two-proportion z-test** to determine whether the observed difference in completion rates is statistically significant.

**Why a z-test?** With sample sizes of 1,500 per group, the Central Limit Theorem guarantees that the sampling distribution of a proportion is approximately normal. The z-test exploits this to compute exact probabilities.

## Task 3.1: Calculate the Pooled Proportion and Standard Error

Under the null hypothesis, both groups have the same true completion rate. We estimate that shared rate using the **pooled proportion**, which combines all completions across both groups.

**Formulas:**

$$p_{pool} = \frac{\text{completions}_A + \text{completions}_B}{n_A + n_B}$$

$$SE = \sqrt{p_{pool} \times (1 - p_{pool}) \times \left(\frac{1}{n_A} + \frac{1}{n_B}\right)}$$

<a name="read-code-5"></a>

**Cell 5 — Calculate Pooled Proportion and Standard Error**

This cell calculates the pooled proportion by combining all completions from both groups and dividing by the total sample size. It then calculates the standard error of the difference under H₀. The standard error tells us how much sampling variability to expect in the observed difference if there were truly no effect.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Pooled proportion (assuming H₀: p_A = p_B)
p_pool = (completions_A + completions_B) / (n_A + n_B)

# Standard error of the difference
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))

print('='*70)
print('POOLED STATISTICS')
print('='*70)
print(f'Pooled proportion: {p_pool:.4f} ({p_pool*100:.2f}%)')
print(f'Standard Error:    {SE:.6f}')
print(f'\nInterpretation: Under H₀ (no true difference), we expect the observed')
print(f'difference to vary by about ±{SE*100:.3f} percentage points due to chance alone.')
print('='*70)

<a name="skip-code-5"></a>

**Expected output:** The pooled proportion (close to 29.75%, the midpoint of 28% and 31.5%) and a standard error around 0.016. A smaller standard error would mean our test is more sensitive to detecting true effects.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

## Task 3.2: Calculate the Test Statistic

The **z-statistic** measures how many standard errors our observed difference is from zero — the value predicted under H₀.

$$z = \frac{p_B - p_A}{SE}$$

A large positive z-statistic means the Treatment group did considerably better than expected by chance.

<a name="read-code-6"></a>

**Cell 6 — Calculate the Z-Statistic**

This cell contains one blank you must fill in: the formula for the z-statistic. It is the observed difference divided by the standard error calculated in Cell 5.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE: fill in the z-statistic formula
z_statistic = ___

print('='*70)
print('TEST STATISTIC')
print('='*70)
print(f'Z-statistic: {z_statistic:.4f}')
print(f'\nInterpretation: The observed difference is {abs(z_statistic):.2f} standard errors')
print(f'  {"above" if z_statistic > 0 else "below"} what we would expect if H₀ were true.')
print('='*70)

<a name="skip-code-6"></a>

**Expected output:** A z-statistic in approximately the range 1.5 to 3.5 (it will vary by seed). If your z-statistic is negative, check that you subtracted in the correct order (p_B minus p_A).

**Conceptual reminder:** A z-statistic of 1.96 means the observed difference is 1.96 standard errors above zero — rare enough that it falls in the 5% tail of the null distribution for a one-tailed test.

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

## Task 3.3: Calculate the P-Value and Critical Value

The **p-value** is the probability of observing a difference this large or larger if H₀ were true. The **critical value** is the threshold the z-statistic must exceed to reject H₀.

Because we are testing whether the Treatment is *better* (not just *different*), this is a **one-tailed (right-tailed) test**.

<a name="read-code-7"></a>

**Cell 7 — Calculate P-Value and Critical Value**

This cell calculates the critical value for a one-tailed test at α = 0.05 using the inverse normal CDF (`norm.ppf`). It then calculates the one-tailed p-value as the area to the right of the z-statistic under the standard normal curve.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
alpha = 0.05

# Critical value (right tail — one-tailed test)
z_critical = norm.ppf(1 - alpha)

# P-value (one-tailed — probability of z this extreme or greater under H₀)
p_value = 1 - norm.cdf(z_statistic)

print('='*70)
print('HYPOTHESIS TEST RESULTS')
print('='*70)
print(f'Significance level (α):  {alpha}')
print(f'Critical value (z):      {z_critical:.4f}')
print(f'Test statistic (z):      {z_statistic:.4f}')
print(f'P-value:                 {p_value:.6f}')
print('\n' + '-'*70)
print('DECISION RULE')
print('-'*70)
print(f'Reject H₀ if p-value < α ({alpha})')
print(f'Reject H₀ if z-statistic > z-critical ({z_critical:.4f})')
print('='*70)

<a name="skip-code-7"></a>

**Expected output:** Critical value of 1.6449 (fixed for α = 0.05, one-tailed) and a p-value that will vary based on your seed. A p-value below 0.05 means you will reject H₀.

**Intuition:** If the p-value is 0.03, it means there is only a 3% chance of observing a difference this large by random chance alone when H₀ is true — rare enough that we conclude the difference is real.

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

## Task 3.4: Visualize the Hypothesis Test

<a name="read-code-8"></a>

**Cell 8 — Plot the Null Distribution with Rejection Region**

This cell draws the standard normal distribution under H₀. It shades the rejection region in red (the right 5% tail), marks the critical value with a red dashed line, marks your test statistic with a green solid line, and shades the p-value area in yellow. The decision (Reject or Fail to Reject H₀) is printed inside the chart.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

x = np.linspace(-4, 4, 1000)
y = norm.pdf(x)

ax.plot(x, y, 'b-', linewidth=2, label='Null Distribution (H₀: p_B = p_A)')

# Rejection region (right tail)
x_rej = x[x >= z_critical]
ax.fill_between(x_rej, norm.pdf(x_rej), alpha=0.35, color='red',
                label=f'Rejection Region (α = {alpha})')
ax.axvline(z_critical, color='red', linestyle='--', linewidth=2,
           label=f'Critical Value = {z_critical:.3f}')

# Test statistic
ax.axvline(z_statistic, color='#8e44ad', linestyle='-', linewidth=3,
           label=f'Test Statistic = {z_statistic:.3f}')

# P-value shading
x_pval = x[x >= z_statistic]
ax.fill_between(x_pval, norm.pdf(x_pval), alpha=0.55, color='yellow',
                label=f'P-value = {p_value:.4f}')

ax.set_xlabel('Z-score', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Hypothesis Test — One-Tailed Z-Test\nMindPath AI Notification Experiment', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(-4, 4)

decision_text  = 'REJECT H₀' if p_value < alpha else 'FAIL TO REJECT H₀'
decision_color = '#27ae60' if p_value < alpha else '#e74c3c'
ax.text(0.5, 0.95, f'Decision: {decision_text}', transform=ax.transAxes,
        fontsize=14, fontweight='bold', va='top', ha='center',
        bbox=dict(boxstyle='round', facecolor=decision_color, alpha=0.3))

plt.tight_layout()
plt.savefig('fig_hypothesis_test.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-8"></a>

**Expected output:** A bell curve with the right tail shaded red (rejection region), a purple vertical line for your test statistic, and a yellow shaded region for the p-value. The decision label appears at the top center of the chart.

**Accessibility note:** Add a Markdown cell below describing this chart. Example: 'The chart shows a standard normal distribution. The red shaded area in the right tail represents the 5% significance level. The purple vertical line (test statistic) falls [inside/outside] the red region, leading to a decision to [reject/fail to reject] H₀.'

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

## Task 3.5: Make Your Decision

<a name="read-code-9"></a>

**Cell 9 — Print the Statistical Decision and Conclusion**

This cell evaluates whether the p-value crosses the α threshold and prints the complete statistical conclusion in plain language, including the specific completion rates observed and the percentage point improvement.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
print('\n' + '='*70)
print('STATISTICAL DECISION')
print('='*70)

if p_value < alpha:
    print('✓ REJECT the null hypothesis')
    print(f'\nReason: p-value ({p_value:.6f}) < α ({alpha})')
    print(f'Also: z-statistic ({z_statistic:.4f}) > critical value ({z_critical:.4f})')
    print('\nConclusion:')
    print(f'There IS statistically significant evidence (at α = {alpha}) that the')
    print(f'AI-personalized notification increases daily check-in completion.')
    print(f'\nTreatment completion rate: {p_B*100:.2f}%')
    print(f'Control completion rate:   {p_A*100:.2f}%')
    print(f'Improvement:               {observed_diff*100:.2f} percentage points')
else:
    print('✗ FAIL TO REJECT the null hypothesis')
    print(f'\nReason: p-value ({p_value:.6f}) ≥ α ({alpha})')
    print(f'Also: z-statistic ({z_statistic:.4f}) ≤ critical value ({z_critical:.4f})')
    print('\nConclusion:')
    print(f'There is NOT statistically significant evidence (at α = {alpha}) that the')
    print(f'AI-personalized notification increases daily check-in completion.')
    print(f'\nWhile Treatment showed {p_B*100:.2f}% vs Control {p_A*100:.2f}%,')
    print(f'this difference could reasonably be due to random sampling variation.')

print('='*70)

<a name="skip-code-9"></a>

**Expected output:** Either a REJECT or FAIL TO REJECT conclusion with the supporting evidence. The specific percentages will reflect your seed's random data.

**Critical point:** Statistical significance tells us the observed difference is unlikely to be due to chance alone. It does not tell us the difference is large enough to matter for the business. That question is addressed in Part 5.

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 4: Confidence Intervals
**Estimated time: 10 minutes**

A p-value tells you whether a result is statistically significant. A **confidence interval** tells you *how big* the effect plausibly is — which is often more useful for business decisions.

The 95% confidence interval is a range of plausible values for the true difference in check-in completion rates. If the interval excludes zero, it agrees with the hypothesis test's rejection of H₀.

## Task 4.1: Calculate the 95% Confidence Interval for the Difference

**Formula:**

$$(p_B - p_A) \pm z^* \times SE$$

where $z^*$ is the critical value for the **two-tailed** 95% interval (1.96).

Note the important distinction: the hypothesis test used a one-tailed critical value (1.645) because we specified a direction. Confidence intervals are always two-sided.

<a name="read-code-10"></a>

**Cell 10 — Calculate the 95% Confidence Interval**

This cell computes the two-tailed critical value (z* = 1.96 for 95% confidence), the margin of error, and the upper and lower bounds of the confidence interval. It also checks whether zero and the 3-percentage-point target are inside or outside the interval.

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
confidence_level = 0.95
z_star = norm.ppf(1 - (1 - confidence_level)/2)  # Two-tailed

margin_of_error = z_star * SE
ci_lower = observed_diff - margin_of_error
ci_upper = observed_diff + margin_of_error

print('='*70)
print('95% CONFIDENCE INTERVAL — DIFFERENCE IN COMPLETION RATES')
print('='*70)
print(f'Observed difference: {observed_diff:.4f} ({observed_diff*100:.2f} percentage points)')
print(f'z* (two-tailed):     {z_star:.4f}')
print(f'Margin of error:     ± {margin_of_error:.4f} ({margin_of_error*100:.3f} pp)')
print(f'\n95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]')
print(f'In percentage points: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]')

print('\nInterpretation:')
print(f'We are 95% confident the true difference in completion rates')
print(f'is between {ci_lower*100:.2f} and {ci_upper*100:.2f} percentage points.')

if ci_lower > 0:
    print('\n✓ The entire interval is above 0, supporting p_B > p_A.')
elif ci_upper < 0:
    print('\n✗ The entire interval is below 0, suggesting p_A > p_B.')
else:
    print('\n⚠️  The interval contains 0 — cannot rule out no true difference.')

target_pp = 3.0
if ci_lower * 100 > target_pp:
    print(f'\n✓ Even the lower bound exceeds the {target_pp}pp target.')
elif ci_upper * 100 >= target_pp:
    print(f'\n⚠️  The interval includes the {target_pp}pp target but also values below it.')
else:
    print(f'\n✗ The interval does not reach the {target_pp}pp business target.')

print('='*70)

<a name="skip-code-10"></a>

**Expected output:** The confidence interval bounds in both decimal and percentage point form, followed by an interpretation of whether the interval excludes zero and whether it includes the 3 percentage point business target.

**Key insight:** If the CI lower bound is above 0 but below 3%, you have statistical significance without practical significance — the effect is real but smaller than the business needs.

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

## Task 4.2: Visualize the Confidence Interval

<a name="read-code-11"></a>

**Cell 11 — Plot the Confidence Interval**

This cell draws a horizontal confidence interval plot. The point estimate (observed difference) is marked with a dot. Two vertical reference lines are added: zero (H₀: no difference) in red and the 3 percentage point business target in green. Annotation shows the exact bounds.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Point estimate and CI
ax.scatter([observed_diff*100], [1], s=200, color='#8e44ad', zorder=5, label='Observed Difference')
ax.plot([ci_lower*100, ci_upper*100], [1, 1], color='#8e44ad', linewidth=5, label='95% Confidence Interval')
ax.plot([ci_lower*100, ci_lower*100], [0.94, 1.06], color='#8e44ad', linewidth=5)
ax.plot([ci_upper*100, ci_upper*100], [0.94, 1.06], color='#8e44ad', linewidth=5)

# Reference lines
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='H₀: No Difference', alpha=0.8)
ax.axvline(3, color='#27ae60', linestyle='--', linewidth=2, label='Target: 3pp Improvement', alpha=0.8)

ax.set_xlabel('Difference in Completion Rates (percentage points)', fontsize=12)
ax.set_title('95% Confidence Interval\nAI-Personalized vs Generic Notification', fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.6)
ax.set_yticks([])
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)

ax.annotate(f'{observed_diff*100:.2f}pp\n[{ci_lower*100:.2f}pp, {ci_upper*100:.2f}pp]',
            xy=(observed_diff*100, 1), xytext=(observed_diff*100, 1.35),
            ha='center', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0e6ff', alpha=0.9),
            arrowprops=dict(arrowstyle='->', lw=2))

plt.tight_layout()
plt.savefig('fig_confidence_interval.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-11"></a>

**Expected output:** A horizontal CI plot with a purple line and dot for the confidence interval. The red dashed line at 0 and the green dashed line at 3pp help you judge practical significance visually.

**Accessibility note:** Add a Markdown cell below describing the chart. Note whether the CI line crosses the red (zero) or green (3pp target) reference lines.

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 5: Business Interpretation and Practical Significance
**Estimated time: 10 minutes**

Statistical significance answers: *Is this difference real?* Practical significance answers: *Is this difference large enough to matter?* A result can be statistically significant but too small to act on — especially with large sample sizes, where even tiny effects can be detected.

## Task 5.1: Statistical vs Practical Significance

**✏️ YOUR TASK:** Answer the following questions in this Markdown cell.

1. **Is the result statistically significant?** (Yes/No and why)

   *[Your answer]*

2. **Does the confidence interval include the 3 percentage point business target?**

   *[Your answer — and what does this mean for the business decision?]*

3. **Based on your analysis, should MindPath deploy the AI-personalized notification system? Why or why not?**

   *[Your answer — consider both statistical and practical significance, and the cost of building and maintaining the AI system]*

4. **What are the potential risks of your recommendation?** Consider Type I error (deploying when there is no real effect) and Type II error (not deploying when there is).

   *[Your answer]*

---
**YOUR ANSWERS ABOVE** — replace the italicized lines.

## Task 5.2: Calculate the Practical Business Impact

<a name="read-code-12"></a>

**Cell 12 — Quantify the Business Impact in Revenue Terms**

This cell translates the statistical difference into business outcomes. It assumes 500,000 monthly active users who receive a notification, and a monthly subscription value of $9.99 per engaged user who completes a check-in (representing improved retention). You can modify these assumptions to explore different scenarios.

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Business assumptions — modify these to explore scenarios
monthly_users = 500000     # Monthly active users who receive a notification
retention_value = 9.99     # Monthly value of an engaged user (subscription)

# Current vs projected completions
current_completions  = monthly_users * p_A
new_completions      = monthly_users * p_B
additional_completions = new_completions - current_completions

# Revenue impact
current_revenue    = current_completions * retention_value
new_revenue        = new_completions * retention_value
additional_monthly = new_revenue - current_revenue
additional_annual  = additional_monthly * 12

print('='*70)
print('BUSINESS IMPACT ANALYSIS — MINDPATH AI NOTIFICATION')
print('='*70)
print(f'\nAssumptions:')
print(f'  Monthly active users receiving notification: {monthly_users:,}')
print(f'  Monthly value per engaged user:            ${retention_value}')
print(f'\nCurrent Performance (Control — Generic):')
print(f'  Completion rate:   {p_A*100:.2f}%')
print(f'  Monthly completions: {current_completions:,.0f}')
print(f'  Monthly revenue:   ${current_revenue:,.0f}')
print(f'\nProjected Performance (Treatment — AI-Personalized):')
print(f'  Completion rate:   {p_B*100:.2f}%')
print(f'  Monthly completions: {new_completions:,.0f}')
print(f'  Monthly revenue:   ${new_revenue:,.0f}')
print(f'\nImpact of Deploying AI-Personalized Notifications:')
print(f'  Additional completions/month: {additional_completions:,.0f}')
print(f'  Additional revenue/month:     ${additional_monthly:,.0f}')
print(f'  Additional revenue/year:      ${additional_annual:,.0f}')
print(f'  Revenue increase:             {(additional_monthly/current_revenue)*100:.2f}%')
print('='*70)

<a name="skip-code-12"></a>

**Expected output:** A full business impact summary showing current performance, projected performance, and the incremental revenue from deploying the AI notification.

**How to use this:** Compare the projected annual revenue gain to the engineering cost of building and maintaining the AI personalization system. If the gain exceeds the cost, the investment is justified — assuming the statistical evidence is strong.

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 6: Power Analysis and Sample Size
**Bonus — complete if time permits**

**Statistical power** is the probability of detecting a true effect when it exists. It equals 1 − β, where β is the Type II error rate.

A power of 80% means that if the true effect is as large as what we observed, we have an 80% chance of detecting it with our sample size. Power below 80% is considered insufficient — you are likely to miss real effects.

<a name="read-code-13"></a>

**Cell 13 — Calculate Statistical Power**

This cell estimates the power of our test given the observed effect size and sample size. It computes the z-score of the critical value under the alternative distribution and uses the normal CDF to find the probability of falling in the rejection region.

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
true_diff = observed_diff

# Under H₁, the critical boundary is at z_critical * SE above zero
# How far is that from the true difference, in SE units?
z_under_alt = (z_critical * SE - true_diff) / SE

power = 1 - norm.cdf(z_under_alt)

print('='*70)
print('STATISTICAL POWER ANALYSIS')
print('='*70)
print(f'Sample size per group:    {n_A:,}')
print(f'Observed effect size:     {observed_diff*100:.2f} percentage points')
print(f'Significance level (α):   {alpha}')
print(f'\nStatistical Power:        {power:.4f} ({power*100:.2f}%)')
print(f'Type II Error Rate (β):   {1-power:.4f} ({(1-power)*100:.2f}%)')
print('\nInterpretation:')
print(f'If the true effect is {observed_diff*100:.2f} percentage points,')
print(f'there is a {power*100:.1f}% probability of detecting it with n={n_A} per group.')

if power >= 0.8:
    print('\n✓ Power meets the standard threshold (≥ 80%)')
else:
    print('\n⚠️  Power is below 80% — consider a larger sample size before concluding the effect is absent.')

print('='*70)

<a name="skip-code-13"></a>

**Expected output:** Power between 40% and 95% (varies by seed). If power is low (below 80%) and your test failed to reject H₀, the null result may reflect insufficient sample size rather than absence of a true effect.

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-14"></a>

**Cell 14 — Required Sample Size for 80% Power**

This cell calculates the minimum sample size per group needed to achieve 80% power for detecting the 3 percentage point business target (28% → 31%) at α = 0.05. It uses the standard sample size formula for comparing two proportions.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Required sample size for 80% power to detect the 3pp business target
from scipy.stats import norm as z_dist

alpha_req  = 0.05
power_req  = 0.80
p1         = 0.28   # Control rate
p2         = 0.31   # Minimum detectable rate (target)

z_alpha = z_dist.ppf(1 - alpha_req)          # One-tailed
z_beta  = z_dist.ppf(power_req)

p_avg   = (p1 + p2) / 2
n_required = (z_alpha * np.sqrt(2 * p_avg * (1 - p_avg))
             + z_beta * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2 / (p2 - p1)**2

print('='*70)
print('REQUIRED SAMPLE SIZE')
print('='*70)
print(f'To detect a {(p2-p1)*100:.0f}pp improvement ({p1*100:.0f}% → {p2*100:.0f}%)')
print(f'at α = {alpha_req}, Power = {power_req*100:.0f}%:')
print(f'\nRequired sample size per group: {int(np.ceil(n_required)):,}')
print(f'Total required sample size:     {int(np.ceil(n_required))*2:,}')
print(f'Your actual sample size/group:  {n_A:,}')

if n_A >= np.ceil(n_required):
    print('\n✓ Your sample size is adequate for the desired power.')
else:
    print(f'\n⚠️  Your test is underpowered. Consider collecting {int(np.ceil(n_required)) - n_A:,} more users per group.')
print('='*70)

<a name="skip-code-14"></a>

**Expected output:** The required sample size per group (approximately 1,700 for these parameters) compared to the actual sample size of 1,500. This tells you whether MindPath ran their test long enough before making a decision.

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 7: Final Report and Recommendations

The ability to communicate statistical findings to non-technical stakeholders is one of the most valued skills in industry. In this part you will write an executive summary suitable for MindPath's product leadership.

## Task 7.1: Executive Summary

**✏️ YOUR TASK:** Write a 3 to 5 sentence executive summary for MindPath's leadership team. This should be written in plain language — no statistical jargon. Your audience understands the business but not hypothesis testing.

Include:
- What was tested and how
- What was found (in plain language)
- Your recommendation
- The expected business impact

**EXECUTIVE SUMMARY:**

*[Write your 3–5 sentence summary here. Replace this line.]*

---
# Reflection Questions

**✏️ YOUR TASK:** Answer each question below in 3–5 sentences. Connect your answers to the MindPath scenario wherever possible.

**1. One-tailed vs two-tailed test**

You ran a one-tailed test because the business question specified a direction (does the AI notification perform *better*?). When would a two-tailed test be more appropriate? How would switching to a two-tailed test change your critical value and p-value?

*[Your answer]*

---

**2. Changing the significance level**

What would happen if MindPath set α = 0.01 instead of 0.05? How does this stricter threshold change the decision? In what product context would a lower α be justified — and when might a higher α (say 0.10) be acceptable?

*[Your answer]*

---

**3. Statistical significance without practical significance**

Suppose the test showed p < 0.05 but the confidence interval was [0.1pp, 0.5pp] — well below the 3pp target. What would you recommend to MindPath? How does this scenario illustrate why reporting only a p-value is insufficient?

*[Your answer]*

---

**4. Confounding variables and threats to validity**

Even with random assignment, what real-world factors could bias this AB test? Think about the nature of a wellness app: seasonality, user tenure, device type, notification permissions, and the novelty effect (users engaging more simply because something changed). How would you design the test to mitigate at least two of these?

*[Your answer]*

---

**5. Test duration and stopping rules**

How long should MindPath run this test before making a decision? What are the risks of stopping too early (peeking) or running the test too long? Research the concept of **sequential testing** or **always-valid p-values** and explain in plain language how they address the peeking problem.

*[Your answer]*

---

**6. Multiple testing and the family-wise error rate (advanced)**

Suppose MindPath runs this experiment and simultaneously tests three other features: a new onboarding flow, a redesigned home screen, and a weekly summary email. If each test uses α = 0.05, what is the probability of at least one false positive across all four tests? What correction method would you apply, and how does it change the individual α thresholds?

*[Your answer]*

---
# Key Concepts Summary

**Hypothesis Testing Framework**
H₀ (null hypothesis) assumes no effect. Hₐ (alternative hypothesis) specifies the expected direction or difference. We reject H₀ only when the p-value falls below α — meaning the observed result is too unlikely under H₀.

**Two-Proportion Z-Test**
Used when comparing binary outcomes (completion/no completion) across two independent groups with large enough samples for the Central Limit Theorem to apply.

**P-Value**
The probability of observing a result this extreme or more extreme if H₀ were true. It is not the probability that H₀ is true.

**Confidence Intervals**
Provide a plausible range for the true effect size. More informative than p-values alone because they communicate magnitude, not just significance.

**Statistical Power**
The probability of correctly rejecting H₀ when a true effect exists. Determined by sample size, effect size, and α. Low power leads to missed real effects (Type II error).

**Practical vs Statistical Significance**
Statistical significance only confirms that an effect is unlikely due to chance. Practical significance requires the effect to be large enough to justify action.

---

## Submission Checklist

Before sharing your link, confirm all of the following:

- Your name is at the top (not your student ID)
- Your random seed is saved (noted in your hypotheses cell or a comment)
- All code cells have been run and contain no unfilled `___` blanks
- Hypotheses are stated clearly (H₀, Hₐ, α, test type) with justification
- Test statistic, p-value, and critical value are calculated and printed
- Confidence interval is calculated and interpreted
- All three visualizations are rendered
- Task 5.1 written questions are answered
- Executive summary is written in plain language
- All six reflection questions are answered
- The file is named `Your_Name.ipynb`

**To share:** Go to **Share** in Colab → set access to **Anyone with the link can view** → copy the link → paste into Canvas.